# Training Node2Vec Embeddings for Research Papers
This notebook trains a Node2Vec model on a citation graph to generate paper embeddings. These embeddings can be used for finding similar papers based on graph structure.

**Instructions for Colab:**
1. Upload your `graph.edgelist` and `metadata.json` files to the session storage (click the folder icon on the left).
2. Run the cells in order.


In [ ]:
import logging

# Reset and force re-config of logging for Jupyter
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    format='%(asctime)s : %(levelname)s : %(message)s',
    level=logging.INFO,
    force=True
)


In [ ]:
# STEP 1: Install and Import Required Libraries
!pip install node2vec networkx

import networkx as nx
from node2vec import Node2Vec
import json
import os
import logging

# 🔥 Enable logging to see training progress from the underlying Gensim model
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)


In [ ]:
# STEP 2: Load Graph from Edgelist
def load_graph(path):
    print(f"Loading graph from {path}...")
    # Node2Vec library works best with an Undirected graph unless specified
    G = nx.read_edgelist(
        path,
        nodetype=int,
        create_using=nx.Graph()
    )

    print("Nodes:", G.number_of_nodes())
    print("Edges:", G.number_of_edges())

    return G

# Check if file exists before running
EDGELIST_PATH = "graph.edgelist"
if os.path.exists(EDGELIST_PATH):
    G = load_graph(EDGELIST_PATH)
else:
    print(f"Error: {EDGELIST_PATH} not found. Please upload it to your Colab workspace.")


In [ ]:
import random
from tqdm import tqdm
from gensim.models import Word2Vec

# STEP 3: Initialize and Train Node2Vec Model (Manual Progress Implementation)
def train_node2vec_manual(G):
    dimensions = 64
    walk_length = 15
    num_walks = 10
    p = 1.0
    q = 1.0
    workers = 2
    
    nodes = list(G.nodes())
    all_walks = []

    print(f"\nPhase 1: Generating {num_walks} random walks per node...")
    
    # Manually iterate through walks with a progress bar
    for i in range(num_walks):
        random.shuffle(nodes)
        for node in tqdm(nodes, desc=f"Walk iteration {i+1}/{num_walks}", leave=False):
            # Simple random walk implementation
            walk = [str(node)]
            while len(walk) < walk_length:
                cur = int(walk[-1])
                neighbors = list(G.neighbors(cur))
                if len(neighbors) > 0:
                    walk.append(str(random.choice(neighbors)))
                else:
                    break
            all_walks.append(walk)

    print(f"\nPhase 2: Training Word2Vec model on {len(all_walks)} walks...")
    
    model = Word2Vec(
        all_walks,
        vector_size=dimensions,
        window=10,
        min_count=1,
        sg=1, # skip-gram
        workers=workers
    )

    print("Training complete.")
    return model

if 'G' in locals():
    model = train_node2vec_manual(G)


In [ ]:
# STEP 4: Save Trained Embeddings
def save_embeddings(model):
    # Word2Vec format works great for many downstream tasks
    model.wv.save_word2vec_format("embeddings.txt")
    print("Saved embeddings -> embeddings.txt")

if 'model' in locals():
    save_embeddings(model)


In [ ]:
# STEP 5: Load Metadata and Perform Similarity Queries
METADATA_PATH = "metadata.json"

def load_metadata(path):
    try:
        with open(path) as f:
            metadata = json.load(f)
        print("Metadata loaded.")
        return metadata
    except Exception as e:
        print(f"Warning: No metadata found at {path} ({e})")
        return {}

def query(model, metadata, paper_id, top_k=5):
    # IDs are typically stored as strings in models
    paper_id = str(paper_id)

    if paper_id not in model.wv:
        print(f"Paper '{paper_id}' not found in trained embeddings.")
        return

    print(f"\nTop {top_k} structurally similar papers to ID {paper_id}:\n")
    results = model.wv.most_similar(paper_id, topn=top_k)

    for pid, score in results:
        # Title lookup using metadata.json
        title = metadata.get(str(pid), {}).get("title", "No Title Found")
        print(f"Score {score:.4f} \u2192 Paper {pid}: {title}")

# Execute query if everything is loaded
metadata = load_metadata(METADATA_PATH)

if 'model' in locals() and 'G' in locals():
    # Test on the first node in the graph
    sample_node = list(G.nodes())[0]
    query(model, metadata, sample_node)
